In [4]:
pip install geopandas rioxarray xarray

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 13.4 MB/s  0:00:00

  Attempting uninstall: xarray

    Found existing installation: xarray 2025.10.1

    Uninstalling xarray-2025.10.1:

      Successfully uninstalled xarray-2025.10.1

   ---------------------------------------- 0/2 [xarray]
   ---------------------------------------- 0/2 [xarray]
   ---------------------------------------- 0/2 [xarray]
   ---------------------------------------- 0/2 [xarray]
   ---------------------------------------- 0/2 [xarray]
   ---------------------------------------- 0/2 [xarray]
   ---------------------------------------- 0/2 [xarray]
   ---------------------------------------- 0/2 [xarray]
   ---------------------------------------- 0/2 [xarray]
   ---------------------------------------- 0/2 [xarray]
   ---------------------------------------- 2/2 [rioxarray]

Note: you may need to restart the kernel to use updated p

In [5]:
"""
fdcf_points.py
==============
Módulo de extração e filtragem de focos de calor (FDCF) a partir de arquivos
NetCDF4 do produto ABI-L2-FDCF do satélite GOES-16.

Fluxo principal:
    1. Recorte espacial na projeção original (evita reprojetar o disco completo).
    2. Reprojeção para SIRGAS 2000 (EPSG:4674).
    3. Filtragem por qualidade: Mask == 10 (foco confirmado) e DQF == 0 (boa qualidade).
    4. Exportação por arquivo: CSV + Shapefile de pontos.
    5. Exportação consolidada: JSON de metadados de toda a execução.

Variáveis do produto FDCF utilizadas:
    - Mask  : classificação do pixel (10 = foco de calor confirmado)
    - DQF   : flag de qualidade (0 = dado válido)
    - Area  : área estimada do foco (m²)
    - Temp  : temperatura do foco (K)
    - Power : potência radiativa do fogo (MW)

Dependências:
    rioxarray, xarray, geopandas, shapely, numpy, pandas

Uso típico:
    processar_pasta_completa(
        pasta_entrada='Dados/ABI-L2-FDCF/netCDF',
        pasta_saida='Arquivos/FDCF_DATA',
    )
"""

import json
import os
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import geopandas as gpd
import numpy as np
import pandas as pd
import rioxarray as rxr
import xarray as xr
from shapely.geometry import Point, box

# ---------------------------------------------------------------------------
# Constantes do módulo
# ---------------------------------------------------------------------------

# Limites geográficos da área de estudo (Pantanal)
PANTANAL_BOUNDS: Dict[str, float] = {
    "xmin": -60.0,   # Longitude oeste
    "xmax": -53.0,   # Longitude leste
    "ymin": -22.0,   # Latitude sul
    "ymax": -15.0,   # Latitude norte
}

# CRS de destino: SIRGAS 2000 geográfico (graus decimais)
CRS_DESTINO = "EPSG:4674"

# CRS de referência geográfica para o bounding box
CRS_GEO = "EPSG:4326"

# Variáveis de interesse no produto FDCF
VARIAVEIS_FDCF = ["Area", "Temp", "Mask", "Power", "DQF"]

# Critérios de filtragem de qualidade
MASK_FOCO_CONFIRMADO = 10   # pixel classificado como foco de calor ativo
DQF_VALIDO = 0              # flag de qualidade indicando dado confiável

# Padrão de data nos nomes de arquivo GOES-16: _sYYYYDDDHHMMSS
_PADRAO_DATA = re.compile(r"_s(\d{4})(\d{3})(\d+)")


# ---------------------------------------------------------------------------
# Funções auxiliares (uso interno)
# ---------------------------------------------------------------------------

def _extrair_data_do_nome(nome_arquivo: str) -> Optional[str]:
    """
    Extrai o identificador de data/hora do nome de um arquivo GOES-16.

    O padrão esperado é `_sYYYYDDDHHMMSS`, onde:
        YYYY = ano, DDD = dia juliano, HHMMSS = hora/minuto/segundo UTC.

    Parâmetros:
        nome_arquivo (str): Nome do arquivo (sem caminho).

    Retorna:
        str | None: String no formato 'YYYYDDDHHMMSS', ou None se não encontrado.

    Exemplos:
        >>> _extrair_data_do_nome('OR_ABI-L2-FDCF-M6_G16_s20201521350164_e(...).nc')
        '20201521350164'
    """
    match = _PADRAO_DATA.search(nome_arquivo)
    if match:
        return f"{match.group(1)}{match.group(2)}{match.group(3)}"
    return None


def _recortar_e_reprojetar(arquivo: Path, bounds: Dict[str, float]) -> xr.Dataset:
    """
    Recorta o raster na projeção original e o reprojeta para o CRS de destino.

    A ordem das operações — recorte antes da reprojeção — é intencional:
    evita reprojetar o disco completo do GOES (5424×5424 pixels) quando apenas
    uma pequena região de interesse é necessária.

    O recorte usa `rio.clip_box` com o bounding box reprojetado para o CRS
    nativo do arquivo, garantindo correteza mesmo quando os eixos x/y não
    são monotônicos.

    Parâmetros:
        arquivo (Path):          Caminho do arquivo NetCDF4.
        bounds  (dict):          Dicionário com xmin, xmax, ymin, ymax em graus.

    Retorna:
        xr.Dataset: Dataset recortado e reprojetado para `CRS_DESTINO`.
    """
    ds = rxr.open_rasterio(arquivo, band_as_variable=False)
    ds = ds.squeeze("band", drop=True)

    if ds.rio.crs:
        # Reprojetar o bounding box geográfico para o CRS nativo do arquivo
        bbox = box(bounds["xmin"], bounds["ymin"], bounds["xmax"], bounds["ymax"])
        bbox_gdf = gpd.GeoDataFrame({"geometry": [bbox]}, crs=CRS_GEO)
        bbox_proj = bbox_gdf.to_crs(ds.rio.crs)

        xmin_p, ymin_p, xmax_p, ymax_p = bbox_proj.total_bounds

        # Recorte via clip_box: robusto para coordenadas não-monotônicas
        ds = ds.rio.clip_box(minx=xmin_p, miny=ymin_p, maxx=xmax_p, maxy=ymax_p)

    # Reprojeção para SIRGAS 2000
    ds = ds.rio.reproject(dst_crs=CRS_DESTINO)

    return ds


def _filtrar_focos(ds: xr.Dataset) -> pd.DataFrame:
    """
    Filtra pixels válidos de foco de calor e retorna um DataFrame.

    Critérios aplicados vetorialmente:
        - Mask == 10  : foco de calor confirmado
        - DQF  == 0   : dado de boa qualidade

    Parâmetros:
        ds (xr.Dataset): Dataset recortado com variáveis Mask e DQF.

    Retorna:
        pd.DataFrame: Registros que satisfazem os critérios de filtragem,
                      com colunas de coordenadas (x, y) e variáveis do produto.
                      DataFrame vazio se Mask ou DQF não estiverem presentes.
    """
    if "Mask" not in ds or "DQF" not in ds:
        print("  ⚠️  Variáveis 'Mask' ou 'DQF' não encontradas no dataset.")
        return pd.DataFrame()

    mascara_valida = (ds["Mask"].values == MASK_FOCO_CONFIRMADO) & \
                     (ds["DQF"].values == DQF_VALIDO)

    # Aplica filtro e descarta linhas/colunas inteiramente NaN
    ds_filtrado = ds.where(mascara_valida)
    ds_filtrado = ds_filtrado.dropna(dim="x", how="all")
    ds_filtrado = ds_filtrado.dropna(dim="y", how="all")

    return ds_filtrado.to_dataframe().reset_index()


def _salvar_shapefile(df: pd.DataFrame, caminho: Path) -> Optional[gpd.GeoDataFrame]:
    """
    Salva um DataFrame de focos como shapefile de pontos.

    Parâmetros:
        df      (pd.DataFrame): DataFrame com colunas 'lon' e 'lat'.
        caminho (Path):         Caminho completo do arquivo de saída.

    Retorna:
        GeoDataFrame | None: GeoDataFrame salvo, ou None se o DataFrame estiver vazio.
    """
    if df.empty:
        print(f"  ⚠️  Sem dados para salvar como shapefile: {caminho.name}")
        return None

    geometry = [Point(lon, lat) for lon, lat in zip(df["lon"], df["lat"])]
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=CRS_DESTINO)
    gdf.to_file(caminho)
    print(f"  🗺️  Shapefile salvo: {caminho.name}")
    return gdf


# ---------------------------------------------------------------------------
# Funções públicas
# ---------------------------------------------------------------------------

def processar_arquivo(
    arquivo: Path,
    pasta_saida: Path,
    bounds: Dict[str, float] = PANTANAL_BOUNDS,
) -> Tuple[bool, Dict]:
    """
    Processa um único arquivo NetCDF4 do produto FDCF.

    Pipeline por arquivo:
        1. Extração do identificador de data/hora do nome do arquivo.
        2. Recorte espacial e reprojeção.
        3. Seleção das variáveis de interesse disponíveis.
        4. Filtragem por qualidade.
        5. Salvamento: CSV + Shapefile.

    Parâmetros:
        arquivo     (Path): Caminho do arquivo NetCDF4.
        pasta_saida (Path): Diretório de saída para CSV e Shapefile.
        bounds      (dict): Limites geográficos de recorte (padrão: Pantanal).

    Retorna:
        tuple:
            bool: True se processado com sucesso, False caso contrário.
            dict: Metadados do arquivo processado.
    """
    meta = {
        "arquivo_original": arquivo.name,
        "data_identificador": None,
        "data_processamento": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        "numero_registros": 0,
        "variaveis_processadas": [],
        "erro": None,
    }

    try:
        print(f"  → Processando: {arquivo.name}")

        # 1. Identificador de data
        data_id = _extrair_data_do_nome(arquivo.name)
        if data_id is None:
            data_id = arquivo.stem
            print(f"  ⚠️  Padrão de data não encontrado, usando: {data_id}")
        else:
            print(f"  📅 Data extraída: {data_id}")
        meta["data_identificador"] = data_id

        # 2. Recorte e reprojeção
        ds = _recortar_e_reprojetar(arquivo, bounds)

        # 3. Seleciona variáveis disponíveis
        variaveis = [v for v in VARIAVEIS_FDCF if v in ds.data_vars]
        ds = ds[variaveis]
        meta["variaveis_processadas"] = variaveis

        # 4. Adiciona timestamp dos atributos do arquivo, se disponível
        if "time_coverage_start" in ds.attrs:
            ds.attrs["time"] = pd.to_datetime(ds.attrs["time_coverage_start"])

        # 5. Filtragem
        df = _filtrar_focos(ds)

        # Renomeia coordenadas e variáveis para nomes padronizados
        df = df.rename(columns={"x": "lon", "y": "lat", "Area": "Area_m2", "Temp": "Temp_K"})

        # Adiciona coluna de tempo se disponível nos atributos
        if "time_coverage_start" in ds.attrs:
            df["time"] = pd.to_datetime(ds.attrs["time_coverage_start"])

        meta["numero_registros"] = len(df)

        # 6. Exportação
        csv_path = pasta_saida / f"dados_filtrados_{data_id}.csv"
        shp_path = pasta_saida / f"focos_{data_id}.shp"

        df.to_csv(csv_path, index=False, encoding="utf-8")
        print(f"  💾 CSV salvo: {csv_path.name}")

        _salvar_shapefile(df, shp_path)

        print(f"  ✅ Processado com sucesso! ({len(df)} registros)")
        return True, meta

    except Exception as e:
        msg = str(e)
        print(f"  ❌ Erro: {msg}")
        meta["erro"] = msg
        return False, meta


def processar_pasta_completa(
    pasta_entrada: str | Path,
    pasta_saida: Optional[str | Path] = None,
    bounds: Dict[str, float] = PANTANAL_BOUNDS,
    extensoes: Optional[List[str]] = None,
) -> None:
    """
    Processa todos os arquivos NetCDF4 de uma pasta e salva os resultados.

    Ao final, gera um único arquivo JSON consolidado com os metadados de
    todos os arquivos processados (sucesso e erro).

    Parâmetros:
        pasta_entrada (str | Path): Diretório contendo os arquivos `.nc`.
        pasta_saida   (str | Path): Diretório de saída (padrão: `pasta_entrada/resultados`).
        bounds        (dict):       Limites geográficos de recorte (padrão: Pantanal).
        extensoes     (list[str]):  Extensões aceitas (padrão: ['.nc']).

    Levanta:
        FileNotFoundError: Se `pasta_entrada` não existir.
    """
    pasta_entrada = Path(pasta_entrada)
    if not pasta_entrada.exists():
        raise FileNotFoundError(f"Pasta de entrada não encontrada: '{pasta_entrada}'")

    pasta_saida = Path(pasta_saida) if pasta_saida else pasta_entrada / "resultados"
    pasta_saida.mkdir(parents=True, exist_ok=True)

    extensoes = extensoes or [".nc"]

    # Lista e filtra arquivos pela extensão
    arquivos = sorted(
        f for f in pasta_entrada.iterdir()
        if f.is_file() and f.suffix.lower() in extensoes
    )

    if not arquivos:
        print(f"⚠️  Nenhum arquivo com extensões {extensoes} encontrado em '{pasta_entrada}'")
        return

    print(f"📁 Entrada : {pasta_entrada}")
    print(f"📁 Saída   : {pasta_saida}")
    print(f"📊 Arquivos: {len(arquivos)}")
    print("=" * 60)

    # Processamento
    todos_metadados = []
    sucesso = 0

    for idx, arquivo in enumerate(arquivos, start=1):
        print(f"\n[{idx}/{len(arquivos)}] {arquivo.name}")
        print("-" * 40)

        ok, meta = processar_arquivo(arquivo, pasta_saida, bounds)
        todos_metadados.append(meta)
        if ok:
            sucesso += 1

    # Metadados consolidados — um único JSON para toda a execução
    resumo = {
        "data_execucao": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        "pasta_entrada": str(pasta_entrada),
        "pasta_saida": str(pasta_saida),
        "bounds_utilizados": bounds,
        "total_arquivos": len(arquivos),
        "processados_com_sucesso": sucesso,
        "processados_com_erro": len(arquivos) - sucesso,
        "arquivos": todos_metadados,
    }

    meta_path = pasta_saida / "metadados_execucao.json"
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(resumo, f, indent=4, ensure_ascii=False, default=str)

    # Resumo final
    print("\n" + "=" * 60)
    print("📈 RESUMO DO PROCESSAMENTO:")
    print(f"  ✅ Sucesso : {sucesso}/{len(arquivos)}")
    print(f"  ❌ Erros   : {len(arquivos) - sucesso}/{len(arquivos)}")
    print(f"  📋 Metadados consolidados: {meta_path.name}")
    print(f"  💾 Resultados em: {pasta_saida}")
    print("=" * 60)


# ---------------------------------------------------------------------------
# Utilitários de diagnóstico
# ---------------------------------------------------------------------------

def testar_extracao_data(nomes: Optional[List[str]] = None) -> None:
    """
    Testa a extração de data/hora a partir de nomes de arquivos GOES-16.

    Parâmetros:
        nomes (list[str] | None): Nomes de arquivo para testar.
                                  Se None, usa exemplos embutidos.
    """
    if nomes is None:
        nomes = [
            "OR_ABI-L2-FDCF-M6_G16_s20201521350164_e20201521359472_c20201521400037(1).nc",
            "OR_ABI-L2-FDCF-M6_G16_s20201521350164_e20201521359472_c20201521400037.nc",
            "arquivo_s2023001123456_test.nc",
            "sem_padrao.nc",
        ]

    print("Teste de extração de data:")
    print("-" * 50)
    for nome in nomes:
        data = _extrair_data_do_nome(nome)
        status = f"✅ {data}" if data else "⚠️  Não encontrado"
        print(f"  {nome}\n  → {status}\n")


# ---------------------------------------------------------------------------
# Exemplo de uso
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    PASTA_ENTRADA = Path("Dados") / "ABI-L2-FDCF" / "netCDF"
    PASTA_SAIDA   = Path("Arquivos") / "FDCF_DATA"

    processar_pasta_completa(
        pasta_entrada=PASTA_ENTRADA,
        pasta_saida=PASTA_SAIDA,
    )

📁 Entrada : Dados\ABI-L2-FDCF\netCDF
📁 Saída   : Arquivos\FDCF_DATA
📊 Arquivos: 189

[1/189] OR_ABI-L2-FDCF-M6_G16_s20201822000210_e20201822009518_c20201822010140.nc
----------------------------------------
  → Processando: OR_ABI-L2-FDCF-M6_G16_s20201822000210_e20201822009518_c20201822010140.nc
  📅 Data extraída: 20201822000210
  ❌ Erro: Invalid version: 'unknown'

[2/189] OR_ABI-L2-FDCF-M6_G16_s20201822010210_e20201822019518_c20201822020106.nc
----------------------------------------
  → Processando: OR_ABI-L2-FDCF-M6_G16_s20201822010210_e20201822019518_c20201822020106.nc
  📅 Data extraída: 20201822010210
  ❌ Erro: Invalid version: 'unknown'

[3/189] OR_ABI-L2-FDCF-M6_G16_s20201822020210_e20201822029518_c20201822030061.nc
----------------------------------------
  → Processando: OR_ABI-L2-FDCF-M6_G16_s20201822020210_e20201822029518_c20201822030061.nc
  📅 Data extraída: 20201822020210
  ❌ Erro: Invalid version: 'unknown'

[4/189] OR_ABI-L2-FDCF-M6_G16_s20201822030210_e20201822039517_c